# OC20 200K reconnaissance

This notebook is a read-only exploration aid for the local OC20 S2EF 200K training subset. It visualises one structure, inspects its paired sidecar row, and joins the record to the supplied mappings.

It deliberately does **not** implement the reusable profiler. Reusable discovery, parsing, and validation logic belongs in `src/` with tests. Raw data is never modified.

## Setup

Run JupyterLab from the repository root with `uv run jupyter lab`. The `.ipynb` is also compatible with a separately installed classic Jupyter Notebook interface. The notebook expects the locally downloaded OC20 data in `data/raw/`.

In [ ]:
from __future__ import annotations

import lzma
import pickle
from io import StringIO
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
from ase.visualize.plot import plot_atoms


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the repository root. Open this notebook from inside the project."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())
RAW_ROOT = REPOSITORY_ROOT / "data" / "raw"
S2EF_ROOT = RAW_ROOT / "s2ef_train_200K" / "s2ef_train_200K"
METADATA_PATH = RAW_ROOT / "oc20_data_mapping.pkl"
RELATIONSHIPS_PATH = RAW_ROOT / "mapping_adslab_slab.pkl"

if not S2EF_ROOT.is_dir():
    raise FileNotFoundError(
        "OC20 raw data was not found. Start JupyterLab from the repository root "
        "after placing the selected files under data/raw/."
    )

## Inventory the paired shards

The 200K subset should contain matching `N.extxyz.xz` structure shards and `N.txt.xz` sidecar shards. The sidecar row at a given ordinal position belongs to the structure at the same ordinal position.

In [ ]:
def shard_id(path: Path) -> int:
    return int(path.name.split(".", maxsplit=1)[0])


structure_shards = sorted(S2EF_ROOT.glob("*.extxyz.xz"), key=shard_id)
sidecar_shards = sorted(S2EF_ROOT.glob("*.txt.xz"), key=shard_id)
structure_ids = {shard_id(path) for path in structure_shards}
sidecar_ids = {shard_id(path) for path in sidecar_shards}


def count_sidecar_rows(path: Path) -> int:
    with lzma.open(path, mode="rt") as file:
        return sum(1 for _ in file)


selected_frame_count = sum(count_sidecar_rows(path) for path in sidecar_shards)

print(f"Structure shards: {len(structure_shards)}")
print(f"Sidecar shards:   {len(sidecar_shards)}")
print(f"Paired IDs:       {len(structure_ids & sidecar_ids)}")
print(f"Missing sidecars: {sorted(structure_ids - sidecar_ids)}")
print(f"Missing structures: {sorted(sidecar_ids - structure_ids)}")
print(f"Selected frames:  {selected_frame_count:,}")

## Inspect one paired record

Choose a shard and an ordinal record within it. The notebook reads only that one extended-XYZ record, so it does not load all 5,000 structures in a shard into memory.

In [ ]:
SHARD_ID = 0
RECORD_INDEX = 0
RAW_PREVIEW_ATOM_ROWS = 6


def read_extxyz_record(
    path: Path, record_index: int
) -> tuple[str, int, str, list[str]]:
    if record_index < 0:
        raise ValueError("RECORD_INDEX must be zero or greater.")

    with lzma.open(path, mode="rt") as file:
        for current_index in range(record_index + 1):
            atom_count_line = file.readline()
            if not atom_count_line:
                raise IndexError(f"Record {record_index} is outside shard: {path.name}")
            atom_count = int(atom_count_line)
            header_line = file.readline()
            atom_lines = list(islice(file, atom_count))

    if len(atom_lines) != atom_count:
        raise ValueError(f"Incomplete extended-XYZ record in: {path}")

    record_text = atom_count_line + header_line + "".join(atom_lines)
    return record_text, atom_count, header_line, atom_lines


def read_sidecar_row(path: Path, record_index: int) -> tuple[str, str, float]:
    with lzma.open(path, mode="rt") as file:
        row = next(islice(file, record_index, record_index + 1), "").strip()
    if not row:
        raise IndexError(f"Row {record_index} is outside sidecar: {path.name}")
    system_id, frame_number, reference_energy = row.split(",")
    return system_id, frame_number, float(reference_energy)


structure_path = S2EF_ROOT / f"{SHARD_ID}.extxyz.xz"
sidecar_path = S2EF_ROOT / f"{SHARD_ID}.txt.xz"
record_text, atom_count, header_line, atom_lines = read_extxyz_record(
    structure_path, RECORD_INDEX
)
atoms = read(StringIO(record_text), format="extxyz")
system_id, frame_number, reference_energy = read_sidecar_row(sidecar_path, RECORD_INDEX)

raw_energy = atoms.get_potential_energy()
max_force = np.linalg.norm(atoms.get_forces(), axis=1).max()

preview = [str(atom_count), header_line.rstrip()] + [
    line.rstrip() for line in atom_lines[:RAW_PREVIEW_ATOM_ROWS]
]
print(f"Raw structure text from {structure_path.name}:")
print("\n".join(preview))
if atom_count > RAW_PREVIEW_ATOM_ROWS:
    print(f"... {atom_count - RAW_PREVIEW_ATOM_ROWS} additional atom rows omitted")

print(f"\nFields read from {sidecar_path.name}:")
print(f"  System ID:        {system_id}")
print(f"  Source frame:     {frame_number}")
print(f"  Reference energy: {reference_energy:.6f} eV")

print(f"\nFields parsed or derived from {structure_path.name}:")
print(f"  Shard / ordinal:  {SHARD_ID} / {RECORD_INDEX}")
print(f"  Chemical formula: {atoms.get_chemical_formula()}")
print(f"  Atom count:       {len(atoms)}")
print(f"  Raw frame energy: {raw_energy:.6f} eV")
print(f"  Maximum force:    {max_force:.6f} eV/A")
print(f"  Periodic boundary:{atoms.pbc.tolist()}")
print(f"  Atom properties:  {sorted(atoms.arrays)}")

The raw frame energy and sidecar reference energy are separate source fields. This notebook displays both but does not calculate an adsorption energy or infer one from their difference.

## Visualise the structure

This is a static scientific view suitable for notebooks and reports. The periodic cell is drawn around the atoms. For interactive rotation, use ASE GUI as documented in `docs/design/oc20-data-understanding.md`.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 7))
plot_atoms(atoms, ax=ax, rotation="35x,0y,45z", show_unit_cell=2)
ax.set_axis_off()
ax.set_title(f"{system_id}: {atoms.get_chemical_formula()}")
fig.tight_layout()
plt.show()

## Join the OC20 mappings

Pickle files can execute code during loading. Only run this cell after verifying that these local files came from the official OC20 release and match its published checksums.

In [ ]:
with METADATA_PATH.open("rb") as file:
    metadata = pickle.load(file)

with RELATIONSHIPS_PATH.open("rb") as file:
    relationships = pickle.load(file)

record_metadata = metadata[system_id]
clean_slab_system_id = relationships.get(system_id)

print("Dataset scope:")
print(f"  Downloaded S2EF structures: {selected_frame_count:,}")
print(f"  Metadata system IDs:         {len(metadata):,}")
print(f"  Clean-slab relationships:    {len(relationships):,}")
print(f"  Metadata IDs without a relationship: {len(metadata) - len(relationships):,}")
print()

for field in (
    "ads_symbols",
    "bulk_symbols",
    "bulk_mpid",
    "miller_index",
    "adsorption_site",
    "anomaly",
    "split",
    "top",
    "shift",
):
    print(f"{field:16} {record_metadata[field]}")

if clean_slab_system_id is None:
    print(f"{'clean_slab_system':16} not available in this mapping")
else:
    print(f"{'clean_slab_system':16} {clean_slab_system_id}")

## Findings to carry into the profiler

- Pair structure and sidecar shards by their numeric stem.
- Pair an individual structure and sidecar row by ordinal position within the pair.
- Preserve raw energy, reference energy, force data, movement constraints, mappings, and anomaly flags as separate evidence.
- Treat adjacent records in this S2EF subset as independent systems, not a playback sequence.
- Keep this notebook as a visual check; put reusable logic and tests in the package.